# Python descriptors: a second, advanced problem-solving tutorial

## From small experiments to robust instance-property designs

**A standalone companion notebook; no earlier notebook needs to be run.**

This notebook deliberately uses the small-step teaching format of the supplied notes: pose a question, predict the outcome, execute one short experiment, inspect the evidence, explain it, and only then change the implementation. There are **14 new multi-stage problems with worked solutions**, smaller sub-experiments, counterexamples, and assertions.

**Topic:** descriptors as instance properties; storage location, binding, lifetime, weak references, attribute lookup, and design constraints.

**How to use it:** run from the first cell downward in a fresh Python 3 kernel. At each *Prediction* prompt, decide what you expect before executing the next code cell. The last cell checks the principal results. No third-party packages or unsafe inspection of memory addresses are required.

## Before we begin: our small testing utility

A teaching notebook benefits from precise failure tests. Instead of intentionally leaving a cell with an uncaught exception (which would interrupt **Run All**), we will catch the *specific* error and assert that it occurred. We will not silently catch every exception.

In [1]:
from contextlib import contextmanager
import gc
import weakref

@contextmanager
def expecting(error_type):
    """Assert that the wrapped statement raises precisely this kind of error."""
    try:
        yield
    except error_type as exc:
        print(f"Expected {type(exc).__name__}: {exc}")
    else:
        raise AssertionError(f"Expected {error_type.__name__}, but nothing was raised")

print("Experiment helper ready.")

Experiment helper ready.


A recurring rule: **a descriptor instance belongs to a class, but each managed instance needs its own value**. We will repeatedly examine where references and names actually live instead of assuming the descriptor behaves like an ordinary attribute.

---

# Problem 1 — Attaching a descriptor *after* class creation

We know that descriptors can learn their assigned name using `__set_name__`. But **when** does Python call that method?

**Tasks**
1. Create an owner class first and attach a descriptor afterward.
2. Observe whether `__set_name__` runs automatically.
3. Repair late binding explicitly.
4. Contrast it with binding during the class statement.

### Step 1 — Instrument the binding hook

We first build a deliberately tiny descriptor. Its hook records a `(class, attribute)` binding. We will not store a value yet; first we want to isolate the class-creation event.

In [2]:
class BindingProbe:
    def __init__(self):
        self.events = []

    def __set_name__(self, owner, name):
        self.events.append((owner.__name__, name))

    def __get__(self, instance, owner=None):
        return self if instance is None else tuple(self.events)


class Early:
    field = BindingProbe()

print("Early.field events:", Early.field.events)
assert Early.field.events == [("Early", "field")]

Early.field events: [('Early', 'field')]


Python invoked `__set_name__` while constructing `Early`. Now we will create `Late` **without** the descriptor and assign to its class afterward. Predict whether `setattr` also invokes the hook.

In [3]:
class Late:
    pass

late_descriptor = BindingProbe()
Late.field = late_descriptor
print("After late assignment:", late_descriptor.events)
assert late_descriptor.events == []

After late assignment: []


### Step 2 — Repair the late installation

The descriptor protocol still works when installed later, but the automatic *notification* does not happen. A framework that installs descriptors dynamically must call the hook itself when needed.

In [4]:
late_descriptor.__set_name__(Late, "field")
print("After explicit hook:", Late().field)
assert Late().field == (("Late", "field"),)

After explicit hook: (('Late', 'field'),)


**Solution:** prefer descriptors in the original class namespace, or arrange the same initialization explicitly for dynamic installation. Note that calling a hook twice is not guaranteed to be safe for arbitrary descriptor implementations. Only call it when appropriate.

---

# Problem 2 — The surprisingly recursive setter

A descriptor wants to write its value on an instance. A tempting implementation is `setattr(instance, self.name, value)`.

**Tasks:** demonstrate the recursive call path safely, then choose a different storage location. Explain why this changes the behavior.

### Step 1 — Build a controlled failure

Instead of exhausting Python's call stack, we count calls and deliberately stop after three recursive entries. This makes the failure predictable.

In [5]:
class RecursiveSetter:
    def __set_name__(self, owner, name):
        self.name = name
        self.calls = 0

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        self.calls += 1
        if self.calls == 3:
            raise RuntimeError("recursive descriptor assignment detected")
        setattr(instance, self.name, value)  # invokes this __set__ again


class LoopExample:
    number = RecursiveSetter()

with expecting(RuntimeError):
    LoopExample().number = 7

print("Descriptor setter calls:", LoopExample.number.calls)
assert LoopExample.number.calls == 3

Expected RuntimeError: recursive descriptor assignment detected
Descriptor setter calls: 3


### Step 2 — Why did it recurse?

Assignment to `obj.number` resolves the data descriptor named `number` and calls its `__set__`. The setter then assigns to **exactly the same public name**, triggering itself again. We must choose storage that does not route back through this descriptor.

One straightforward answer for ordinary instances is a unique private instance-dictionary key.

In [6]:
class DictionaryNumber:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"__value_for_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = int(value)


class WorkingExample:
    number = DictionaryNumber()

item = WorkingExample()
item.number = "17"
print("Read:", item.number, "| instance dictionary:", vars(item))
assert item.number == 17
assert "number" not in vars(item)

Read: 17 | instance dictionary: {'__value_for_number': 17}


**Solution:** write to a different key. Direct `instance.__dict__` access intentionally requires a normal instance dictionary; it is not suitable for every slotted class. We will investigate storage constraints later.

---

# Problem 3 — Is a missing value the same as `None`?

Suppose `None` is a legitimate value for a descriptor-managed attribute. A getter that uses `dict.get(key)` cannot tell whether the attribute was never set or was explicitly set to `None`.

**Tasks:** exhibit the ambiguity; implement `__get__` and `__delete__` using a distinct missing state; ensure `hasattr` behaves naturally after deletion.

### Step 1 — See the ambiguity

Predict the outputs before executing: `get` returns `None` for both a missing key and a stored `None` when no explicit default is supplied.

In [7]:
values = {}
print("Missing:", values.get("value"))
values["value"] = None
print("Present with None:", values.get("value"))
print("Actually present?", "value" in values)
assert values.get("value") is None
assert "value" in values

Missing: None
Present with None: None
Actually present? True


### Step 2 — Make absence explicit

A unique sentinel object represents *missing*, while `None` remains an ordinary stored value. The class-access branch returns the descriptor rather than reading instance state.

In [8]:
MISSING_3 = object()

class OptionalValue:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        value = instance.__dict__.get(self.key, MISSING_3)
        if value is MISSING_3:
            raise AttributeError(f"{self.name} has not been assigned")
        return value

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

    def __delete__(self, instance):
        if self.key not in instance.__dict__:
            raise AttributeError(f"{self.name} has not been assigned")
        del instance.__dict__[self.key]


class Message:
    note = OptionalValue()

msg = Message()
print("Before assignment, hasattr:", hasattr(msg, "note"))
assert not hasattr(msg, "note")

msg.note = None
print("After storing None, hasattr:", hasattr(msg, "note"))
assert hasattr(msg, "note") and msg.note is None

Before assignment, hasattr: False
After storing None, hasattr: True


### Step 3 — Deletion is a state transition, not assignment to `None`

If deletion merely assigned `None`, callers could not distinguish a deliberately empty value from an uninitialized one. Our `__delete__` truly removes the key.

In [9]:
del msg.note
print("After deletion:", vars(msg))
assert not hasattr(msg, "note")
with expecting(AttributeError):
    _ = msg.note
with expecting(AttributeError):
    del msg.note

After deletion: {}
Expected AttributeError: note has not been assigned
Expected AttributeError: note has not been assigned


**Solution:** represent missing values explicitly, and let a missing read raise `AttributeError`. This matters for `hasattr`, serialization, defaults, and diagnostic tools.

---

# Problem 4 — A getter raises `AttributeError` — but what does the caller see?

An `AttributeError` raised inside a descriptor can cause an object's `__getattr__` fallback to run. This means a fallback may hide a genuine descriptor initialization error.

**Tasks:** trace the actual order of calls and distinguish direct protocol access from normal attribute access.

### Step 1 — Record the events

We intentionally use an uninitialized getter that raises `AttributeError`. The owning class implements `__getattr__`, which Python uses when ordinary attribute lookup fails.

In [10]:
class MissingDescriptor:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        instance.events.append("descriptor.__get__")
        raise AttributeError("uninitialized managed field")


class FallbackOwner:
    field = MissingDescriptor()

    def __init__(self):
        self.events = []

    def __getattr__(self, name):
        self.events.append(f"__getattr__({name})")
        return f"fallback for {name}"


owner4 = FallbackOwner()
print("Normal lookup:", owner4.field)
print("Call order:", owner4.events)
assert owner4.events == ["descriptor.__get__", "__getattr__(field)"]

Normal lookup: fallback for field
Call order: ['descriptor.__get__', '__getattr__(field)']


### Step 2 — Compare a direct descriptor call

Calling the raw descriptor's `__get__` directly bypasses the owner's normal lookup machinery. The exception now surfaces unchanged.

In [11]:
raw4 = vars(FallbackOwner)["field"]
with expecting(AttributeError):
    raw4.__get__(owner4, FallbackOwner)
print("Events now:", owner4.events)

Expected AttributeError: uninitialized managed field
Events now: ['descriptor.__get__', '__getattr__(field)', 'descriptor.__get__']


**Solution:** `AttributeError` has a specific meaning to attribute lookup. It is appropriate for a truly missing value, but an owning class with `__getattr__` may intentionally supply a fallback. If absence must not be masked, document the policy or design the owner-level fallback to handle managed names explicitly.

---

# Problem 5 — A private storage key is not actually collision-proof

`__set_name__` commonly produces a key such as `"_" + name`. This removes a collision between *different public descriptor names*, but it can still collide with an ordinary attribute that already uses `_name`.

**Tasks:** construct the conflict and establish a class design rule for avoiding it.

### Step 1 — A perfectly ordinary conflict

A class has its own internal `_code` attribute. The descriptor uses that same key for its managed `code`. We will show the descriptor overwriting the class's internal state.

In [12]:
class PrefixField:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value


class Conflicting:
    code = PrefixField()

    def __init__(self):
        self._code = "internal status"


conflict = Conflicting()
print("Before:", vars(conflict))
conflict.code = "client code"
print("After:", vars(conflict))
assert conflict._code == "client code"

Before: {'_code': 'internal status'}
After: {'_code': 'client code'}


### Step 2 — Reserve internal keys deliberately

A practical design is to **reserve and document** a storage namespace. An unusual key reduces accidental collisions, but is not a mathematical guarantee. The owner should not use that namespace for unrelated state.

In [13]:
class ReservedKeyField:
    def __set_name__(self, owner, name):
        self.key = f"__managed_field__{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value


class NonConflicting:
    code = ReservedKeyField()

    def __init__(self):
        self._code = "internal status"


safe5 = NonConflicting()
safe5.code = "client code"
print(vars(safe5))
assert safe5._code == "internal status"
assert safe5.code == "client code"

{'_code': 'internal status', '__managed_field__code': 'client code'}


**Solution:** class-local naming conventions are an interface contract. For a controlled object model, private instance storage is simple; for third-party objects that cannot be modified, external storage may be necessary, with its own limitations.

---

# Problem 6 — A weak-key dictionary can confuse equal *but distinct* objects

Replacing strong-key storage with `weakref.WeakKeyDictionary` prevents the *keys alone* from keeping instances alive. But there is a subtler correctness problem: weak-key dictionary keys use the referents' hashing and equality semantics.

**Tasks:** create two distinct objects that compare equal; observe storage collisions; explain why this is unacceptable for truly per-instance descriptor state.

### Step 1 — Two objects, one equality class

Both accounts have the same account number, so their domain-level equality considers them equal. They are different Python objects, however.

In [14]:
class AccountIdentity:
    def __init__(self, number):
        self.number = number

    def __hash__(self):
        return hash(self.number)

    def __eq__(self, other):
        return isinstance(other, AccountIdentity) and self.number == other.number


a6 = AccountIdentity("001")
b6 = AccountIdentity("001")
print("Same object?", a6 is b6)
print("Compare equal?", a6 == b6)
assert a6 is not b6 and a6 == b6

Same object? False
Compare equal? True


### Step 2 — Use the two objects as weak dictionary keys

Predict whether the mapping has one entry or two after assigning both values.

In [15]:
weak_mapping6 = weakref.WeakKeyDictionary()
weak_mapping6[a6] = "Alice's session"
weak_mapping6[b6] = "Bob's session"
print("Entry count:", len(weak_mapping6))
print("Lookup a:", weak_mapping6[a6])
print("Lookup b:", weak_mapping6[b6])
assert len(weak_mapping6) == 1
assert weak_mapping6[a6] == weak_mapping6[b6] == "Bob's session"

Entry count: 1
Lookup a: Bob's session
Lookup b: Bob's session


### Step 3 — What exactly went wrong?

`WeakKeyDictionary` is weak, but it is not inherently an **identity-keyed** mapping. If `a == b` and their hashes agree, the second assignment may replace the first value. Per-instance descriptor state should not depend on domain equality. One simple solution is to store values inside each object when that is possible.

In [16]:
class LocalSession:
    def __set_name__(self, owner, name):
        self.key = f"__local_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value


class AccountWithSession(AccountIdentity):
    session = LocalSession()


first6 = AccountWithSession("001")
second6 = AccountWithSession("001")
first6.session = "Alice's session"
second6.session = "Bob's session"
print(first6.session, "|", second6.session)
assert first6 == second6
assert first6.session != second6.session

Alice's session | Bob's session


**Solution:** select storage based on the required *identity semantics*, not simply on whether the dictionary is weak. Our next problem develops a weak external store that explicitly uses object identity.

---

# Problem 7 — External instance storage for unhashable objects

Some instances support weak references but are intentionally unhashable (for example, many classes defining `__eq__` without `__hash__`). A `WeakKeyDictionary` will reject them.

**Challenge:** design an external descriptor store that:

- distinguishes objects using identity rather than equality;
- accepts weak-referenceable, unhashable objects;
- does not retain an object merely by storing its reference;
- removes stale entries when the object dies;
- guards the cleanup callback against removing an entry for a different object after identifier reuse.

**Important limitation:** the *stored value* must not itself hold a strong reference back to the managed instance; we investigate that separately.

### Step 1 — Confirm the failure

Python weak references are not the same thing as hashability. The next instance supports `weakref.ref`, but cannot be a key in a normal dictionary or `WeakKeyDictionary`.

In [17]:
class Unhashable7:
    __hash__ = None

    def __eq__(self, other):
        return isinstance(other, Unhashable7)

u7 = Unhashable7()
assert weakref.ref(u7)() is u7
with expecting(TypeError):
    weakref.WeakKeyDictionary()[u7] = 10

Expected TypeError: unhashable type: 'Unhashable7'


### Step 2 — Change the key, and keep a weak identity check

The mapping will use `id(instance)` as an integer key and keep `(weakref, value)` as its entry. An `id` alone is **not enough**: Python may reuse identifiers after objects die. When retrieving, we confirm that the weak reference resolves to the *identical* object. In a cleanup callback, we remove an entry only if it still holds the callback's exact weak reference.

In [18]:
class WeakIdentityField:
    def __init__(self):
        self._entries = {}  # id(instance) -> (weakref, value)

    def __set_name__(self, owner, name):
        self.name = name

    def _key(self, instance):
        return id(instance)

    def _make_ref(self, instance, key):
        def remove(dead_ref):
            # Do not erase a newer entry if this integer identifier was reused.
            record = self._entries.get(key)
            if record is not None and record[0] is dead_ref:
                del self._entries[key]
        return weakref.ref(instance, remove)

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        record = self._entries.get(self._key(instance))
        if record is None or record[0]() is not instance:
            raise AttributeError(f"{self.name} has not been assigned")
        return record[1]

    def __set__(self, instance, value):
        key = self._key(instance)
        record = self._entries.get(key)
        if record is not None and record[0]() is instance:
            self._entries[key] = (record[0], value)
        else:
            self._entries[key] = (self._make_ref(instance, key), value)

    def __delete__(self, instance):
        key = self._key(instance)
        record = self._entries.get(key)
        if record is None or record[0]() is not instance:
            raise AttributeError(f"{self.name} has not been assigned")
        del self._entries[key]

    def tracked(self):
        return len(self._entries)

### Step 3 — Use it with two equal, unhashable instances

The class below defines equality but has no hash. It is intentionally compatible with `weakref.ref` because it has a normal instance layout.

In [19]:
class EquatableUser:
    __hash__ = None
    token = WeakIdentityField()

    def __init__(self, number):
        self.number = number

    def __eq__(self, other):
        return isinstance(other, EquatableUser) and self.number == other.number


u7a = EquatableUser(1)
u7b = EquatableUser(1)
u7a.token = "first"
u7b.token = "second"
print("Equal?", u7a == u7b)
print("Tokens:", u7a.token, u7b.token)
assert u7a == u7b and u7a is not u7b
assert (u7a.token, u7b.token) == ("first", "second")
assert EquatableUser.token.tracked() == 2

Equal? True
Tokens: first second


### Step 4 — Verify weak cleanup and deletion

A weak reference lets us detect whether the object survives after deleting the sole application reference. There is no need to inspect raw addresses or reference counts using low-level memory APIs.

In [20]:
u7a_ref = weakref.ref(u7a)
del u7a
gc.collect()
print("First user alive?", u7a_ref() is not None)
print("External entries:", EquatableUser.token.tracked())
assert u7a_ref() is None
assert EquatableUser.token.tracked() == 1

del u7b.token
assert EquatableUser.token.tracked() == 0
with expecting(AttributeError):
    _ = u7b.token

First user alive? False
External entries: 1
Expected AttributeError: token has not been assigned


**Solution and trade-off:** an identity-keyed weak-reference store handles this narrow problem, but is more complicated than instance storage. It still requires weak-referenceable instances, and a value that references its key can defeat weak cleanup. Prefer simpler instance-side storage when the object design permits it.

---

# Problem 8 — The weak key is weak, but its *value* keeps the instance alive

A weak-key dictionary prevents *its key reference* from retaining the object. However, the mapping strongly owns its **values**. What happens if the value contains a reference back to the very instance used as its key?

**Tasks:** demonstrate a reference cycle rooted in the descriptor's mapping, then repair the design by storing a weak reference within the value.

### Step 1 — Arrange a back-reference

The wrapper `Attachment` owns its `owner` strongly. Our global descriptor object owns the `WeakKeyDictionary`, which strongly owns each `Attachment` value.

In [21]:
class Attachment:
    def __init__(self, owner, data):
        self.owner = owner  # strong reference back to the weak key
        self.data = data


class WeakContainer:
    attachments = weakref.WeakKeyDictionary()


class Managed8:
    pass

item8 = Managed8()
WeakContainer.attachments[item8] = Attachment(item8, "payload")
item8_watch = weakref.ref(item8)
assert len(WeakContainer.attachments) == 1

### Step 2 — Delete the visible instance variable

Predict whether the weak-key mapping becomes empty. The value still points strongly to the instance, so it should not.

In [22]:
del item8
gc.collect()
print("Still alive?", item8_watch() is not None)
print("Mapping size:", len(WeakContainer.attachments))
assert item8_watch() is not None
assert len(WeakContainer.attachments) == 1

Still alive? True
Mapping size: 1


### Step 3 — Release the retained value, then rebuild safely

First clear the demonstration mapping so this notebook does not intentionally hold a leaked test object. Next store `weakref.ref(owner)` inside the value instead of `owner` itself.

In [23]:
WeakContainer.attachments.clear()
gc.collect()
assert item8_watch() is None

class WeakAttachment:
    def __init__(self, owner, data):
        self.owner_ref = weakref.ref(owner)
        self.data = data

class SafeContainer8:
    attachments = weakref.WeakKeyDictionary()

item8b = Managed8()
SafeContainer8.attachments[item8b] = WeakAttachment(item8b, "payload")
item8b_watch = weakref.ref(item8b)
del item8b
gc.collect()
print("Object after safe storage:", item8b_watch())
print("Mapping size after collection:", len(SafeContainer8.attachments))
assert item8b_watch() is None
assert len(SafeContainer8.attachments) == 0

Object after safe storage: None
Mapping size after collection: 0


**Solution:** study the entire reference graph, not just whether the key is weak. A closure, bound method, callback, or value containing `owner` may recreate the same retention problem. No descriptor storage trick can make an object collectible while another reachable object still strongly references it.

---

# Problem 9 — Descriptors meet `@dataclass`: where does the default come from?

`dataclasses` treats descriptor-typed defaults specially. The descriptor is queried through the class during class processing to determine whether there is a field default.

**Tasks:** make a required descriptor-backed dataclass field; see what happens when class access returns a default; verify that generated `__init__` routes through `__set__`. This problem assumes the standard-library `dataclasses` implementation in contemporary Python 3.

### Step 1 — Required field: raise `AttributeError` on class access

This is intentionally different from the common descriptor convention of returning `self` for class access. For a dataclass descriptor field, raising `AttributeError` tells the dataclass machinery that *no default exists*.

In [24]:
from dataclasses import dataclass, fields

class RequiredPositive:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"__dataclass_{name}"
        self.assignments = 0

    def __get__(self, instance, owner=None):
        if instance is None:
            raise AttributeError("no class-level default")
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        self.assignments += 1
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"{self.name} must be a positive integer")
        instance.__dict__[self.key] = value


required_probe9 = RequiredPositive()

@dataclass
class Package9:
    weight: int = required_probe9

package9 = Package9(12)
print(package9, vars(package9))
assert package9.weight == 12
assert required_probe9.assignments == 1
with expecting(ValueError):
    Package9(-2)

Package9(weight=12) {'__dataclass_weight': 12}
Expected ValueError: weight must be a positive integer


### Step 2 — A descriptor-supplied default

Now class access returns an ordinary default value. The generated dataclass `__init__` still assigns the chosen value to the descriptor. Observe how an omitted argument differs from a required field.

In [25]:
class DefaultedPositive:
    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"__defaulted_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return 3  # dataclasses reads this as the field default
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"{self.name} must be a positive integer")
        instance.__dict__[self.key] = value


@dataclass
class Box9:
    height: int = DefaultedPositive()


small9 = Box9()
tall9 = Box9(8)
print("Default:", small9.height, "| Explicit:", tall9.height)
print("Dataclass default:", fields(Box9)[0].default)
assert (small9.height, tall9.height) == (3, 8)
assert fields(Box9)[0].default == 3

Default: 3 | Explicit: 8
Dataclass default: 3


**Solution:** a descriptor used as a dataclass field interacts with dataclass processing *before normal instance reads*. Decide consciously what class access should mean. For general-purpose descriptors, returning `self` is conventional; for descriptor-typed dataclass fields, it can accidentally turn the descriptor object itself into a default. Test the combination rather than assuming decorators are transparent.

---

# Problem 10 — What happens to descriptor values when an instance is copied?

A descriptor with instance-side state and a descriptor with external storage have different copy behavior.

**Tasks:** compare `copy.copy` and `copy.deepcopy` for each design. Explain why a copied object may lose externally stored values.

### Step 1 — Copy ordinary instance-side state

The descriptor lives on the class, but the managed value lives in `__dict__`. Standard object copying typically copies that dictionary. We will store a list to distinguish shallow from deep copying.

In [26]:
import copy

class LocalList:
    def __set_name__(self, owner, name):
        self.key = f"__list_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value


class LocalDocument10:
    tags = LocalList()


original10 = LocalDocument10()
original10.tags = ["python", "descriptors"]
shallow10 = copy.copy(original10)
deep10 = copy.deepcopy(original10)
print("Objects distinct?", original10 is not shallow10, original10 is not deep10)
print("Shallow list shared?", original10.tags is shallow10.tags)
print("Deep list shared?", original10.tags is deep10.tags)
assert shallow10 is not original10 and deep10 is not original10
assert original10.tags is shallow10.tags
assert original10.tags is not deep10.tags

Objects distinct? True True
Shallow list shared? True
Deep list shared? False


### Step 2 — Copy external descriptor-owned state

The next object has no managed value in its instance dictionary. A weak-key descriptor stores the value in a separate table keyed by the original instance. A fresh copy receives its own identity and is not automatically added to that table.

In [27]:
class ExternalField10:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError("external value is unset") from None

    def __set__(self, instance, value):
        self.values[instance] = value


class ExternalDocument10:
    tags = ExternalField10()


outside10 = ExternalDocument10()
outside10.tags = ("python", "descriptors")
copied10 = copy.copy(outside10)
print("Original state:", outside10.tags)
print("Copied instance dictionary:", vars(copied10))
with expecting(AttributeError):
    _ = copied10.tags

Original state: ('python', 'descriptors')
Copied instance dictionary: {}
Expected AttributeError: external value is unset


### Step 3 — Explicitly define copying policy

One reasonable policy is that a copy receives the same immutable external value. We will implement `__copy__` on the owner rather than silently forcing behavior in the descriptor.

In [28]:
class CopyAwareDocument10:
    tags = ExternalField10()

    def __init__(self, tags):
        self.tags = tuple(tags)

    def __copy__(self):
        return type(self)(self.tags)


source10 = CopyAwareDocument10(["a", "b"])
clone10 = copy.copy(source10)
assert clone10 is not source10
assert clone10.tags == source10.tags == ("a", "b")
print("Explicit-copy result:", clone10.tags)

Explicit-copy result: ('a', 'b')


**Solution:** external state is invisible to ordinary instance copying unless you define a contract. The same question arises with serialization and object reconstruction. Keep storage and object lifecycle decisions aligned.

---

# Problem 11 — An instance dictionary can bypass your validation

Data descriptors take precedence during *normal assignment*, but they cannot stop someone from directly editing a writable `__dict__`. We will demonstrate this limit and distinguish convenience validation from a true security boundary.

**Tasks:** validate a field; bypass it; repair it through normal assignment; discuss when a controlled instance layout helps.

### Step 1 — Write the validator

We deliberately use a positive-only value so an invalid state is obvious.

In [29]:
class Positive11:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("expected a number, not bool")
        if value <= 0:
            raise ValueError("must be positive")
        instance.__dict__[self.key] = value


class Invoice11:
    amount = Positive11()


invoice11 = Invoice11()
invoice11.amount = 100
with expecting(ValueError):
    invoice11.amount = -10
assert invoice11.amount == 100

Expected ValueError: must be positive


### Step 2 — Bypass ordinary assignment

Changing the internal storage dictionary directly does not execute the descriptor's setter. This is not a Python bug; normal descriptors are validation mechanisms, not barriers against a caller who can mutate arbitrary internal state.

In [30]:
invoice11.__dict__["_amount"] = -10
print("Value after bypass:", invoice11.amount)
assert invoice11.amount == -10
invoice11.amount = 100  # repair through the normal, validated path
assert invoice11.amount == 100

Value after bypass: -10


### Step 3 — What about `__slots__`?

A slotted owner with no instance dictionary makes this *specific* bypass unavailable, provided the backing slot exists. A caller can still assign the backing slot directly if it is accessible, so this is a design constraint, **not** an authorization or security mechanism.

In [31]:
class SlotPositive11:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else getattr(instance, self.key)

    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError("must be positive")
        setattr(instance, self.key, value)


class SlottedInvoice11:
    __slots__ = ("_amount",)
    amount = SlotPositive11()


slotted11 = SlottedInvoice11()
slotted11.amount = 25
print("Has instance dictionary?", hasattr(slotted11, "__dict__"))
assert not hasattr(slotted11, "__dict__")
assert slotted11.amount == 25

Has instance dictionary? False


**Solution:** descriptors enforce rules for normal attribute operations. Do not treat them as protection against adversarial Python code with direct access to instance internals.

---

# Problem 12 — Mutable managed values: a setter is not an observer

A descriptor validates assignment, but the object returned from `__get__` may itself be mutable. In-place mutation does **not** call `__set__`.

**Tasks:** demonstrate bypass via `.append`, then redesign the field to store an immutable snapshot and require reassignment for changes.

### Step 1 — Validate a list on assignment

We count setter calls to prove that mutating the returned list does not trigger the descriptor.

In [32]:
class WeakList12:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"
        self.setter_calls = 0

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        self.setter_calls += 1
        if not isinstance(value, list) or not all(isinstance(x, str) for x in value):
            raise TypeError("expected a list of strings")
        instance.__dict__[self.key] = value


class Blog12:
    tags = WeakList12()


blog12 = Blog12()
blog12.tags = ["python"]
count_before12 = Blog12.tags.setter_calls
blog12.tags.append(999)  # mutates the existing list; no assignment to blog12.tags
print("Tags:", blog12.tags)
print("Setter called again?", Blog12.tags.setter_calls != count_before12)
assert Blog12.tags.setter_calls == count_before12
assert blog12.tags == ["python", 999]

Tags: ['python', 999]
Setter called again? False


### Step 2 — Normalize to an immutable representation

A tuple prevents `.append` on the managed value. The descriptor also copies from the input iterable so later mutations to the original input list do not change the stored state.

In [33]:
class ImmutableTags12:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        try:
            normalized = tuple(value)
        except TypeError as exc:
            raise TypeError("tags must be an iterable of strings") from exc
        if not all(isinstance(x, str) for x in normalized):
            raise TypeError("tags must contain only strings")
        instance.__dict__[self.key] = normalized


class SafeBlog12:
    tags = ImmutableTags12()

    def __init__(self, tags):
        self.tags = tags

    def add_tag(self, tag):
        self.tags = (*self.tags, tag)  # reuses descriptor validation


incoming12 = ["python"]
safe_blog12 = SafeBlog12(incoming12)
incoming12.append("external change")
print("After input mutation:", safe_blog12.tags)
assert safe_blog12.tags == ("python",)
with expecting(AttributeError):
    safe_blog12.tags.append("descriptors")

After input mutation: ('python',)
Expected AttributeError: 'tuple' object has no attribute 'append'


### Step 3 — Reassign through the public interface

The model's convenience method builds a new immutable value, and the descriptor validates it on assignment. A failed update leaves the old state untouched.

In [34]:
safe_blog12.add_tag("descriptors")
assert safe_blog12.tags == ("python", "descriptors")
with expecting(TypeError):
    safe_blog12.add_tag(123)
assert safe_blog12.tags == ("python", "descriptors")
print("Final validated tags:", safe_blog12.tags)

Expected TypeError: tags must contain only strings
Final validated tags: ('python', 'descriptors')


**Solution:** if invariants must survive ordinary application use, consider immutable snapshots and controlled update methods. This is particularly important for lists, dictionaries, sets, and nested mutable objects.

---

# Problem 13 — Validate first, commit second: assignment should be atomic

An assignment can fail after making a partial change. Good setters normally finish validation **before** changing the instance's state.

**Tasks:** demonstrate partial mutation in a deliberately broken descriptor and repair it; show that the old valid value survives a failed attempt.

### Step 1 — Deliberately broken ordering

This setter commits a new value and *then* validates it. The exception reports a failure, but the invalid value is already stored.

In [35]:
class WrongOrder13:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value  # too early!
        if value < 0:
            raise ValueError("negative balance forbidden")


class BrokenWallet13:
    balance = WrongOrder13()


broken13 = BrokenWallet13()
broken13.balance = 100
with expecting(ValueError):
    broken13.balance = -50
print("Balance *after rejected assignment*:", broken13.balance)
assert broken13.balance == -50

Expected ValueError: negative balance forbidden
Balance *after rejected assignment*: -50


### Step 2 — Perform all checks before committing

The corrected descriptor computes and validates a candidate first. Only after those operations succeed does it update storage.

In [36]:
class ValidateThenCommit13:
    def __set_name__(self, owner, name):
        self.key = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.key]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("balance must be an integer")
        if value < 0:
            raise ValueError("negative balance forbidden")
        instance.__dict__[self.key] = value


class Wallet13:
    balance = ValidateThenCommit13()


wallet13 = Wallet13()
wallet13.balance = 100
with expecting(ValueError):
    wallet13.balance = -50
with expecting(TypeError):
    wallet13.balance = "100"
print("Balance after both failures:", wallet13.balance)
assert wallet13.balance == 100

Expected ValueError: negative balance forbidden
Expected TypeError: balance must be an integer
Balance after both failures: 100


**Solution:** validation should be side-effect-free until commit whenever feasible. For operations that update multiple attributes, single-field descriptors alone do not make the *whole operation* atomic; use an owner-level transaction or coordinated method for that stronger guarantee.

---

# Problem 14 — A complete dependent-property model: cache invalidation, validation, and lifetime

This final exercise integrates several lessons into a **new** use case: a geometric model has two validated dimensions and an expensive derived perimeter calculation. The calculation is cached, but any valid change to a dimension must invalidate it.

Unlike a simple computed or cached descriptor, this design has **dependency-aware invalidation** and avoids keeping instance references in the descriptor. We will build it one small piece at a time.

### Step 1 — Predict the stale-value problem

If a cached result is calculated from width and height, then width changes, what must happen to the previous result? It is now stale. A cached descriptor cannot guess which unrelated attributes affect it; the model must declare that relationship.

Our design: dimensions store validated numbers in the instance dictionary. Their setters remove the cached result on a *successful* update.

In [37]:
class Dimension14:
    def __init__(self, *, invalidates):
        self.invalidates = tuple(invalidates)

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"__dimension_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be numeric")
        if value <= 0:
            raise ValueError(f"{self.name} must be positive")

        # Validate first; then write and invalidate caches.
        instance.__dict__[self.key] = value
        for cache_key in self.invalidates:
            instance.__dict__.pop(cache_key, None)

### Step 2 — Implement a read-only, instance-local cached computation

The descriptor never stores a list of instances. The cache is held by each individual object's dictionary. `__set__` makes it a **data descriptor** so ordinary callers cannot accidentally shadow the public computed attribute.

In [38]:
class CachedPerimeter14:
    def __set_name__(self, owner, name):
        self.name = name
        self.cache_key = f"__cache_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        data = instance.__dict__
        if self.cache_key not in data:
            instance.compute_calls += 1
            data[self.cache_key] = 2 * (instance.width + instance.height)
        return data[self.cache_key]

    def __set__(self, instance, value):
        raise AttributeError(f"{self.name} is a derived, read-only value")


class Rectangle14:
    width = Dimension14(invalidates=("__cache_perimeter",))
    height = Dimension14(invalidates=("__cache_perimeter",))
    perimeter = CachedPerimeter14()

    def __init__(self, width, height):
        self.compute_calls = 0
        self.width = width
        self.height = height

### Step 3 — Verify the happy path

The first read computes; the second read reuses the cached value. Observe the dictionary before and after the first access.

In [39]:
rectangle14 = Rectangle14(3, 4)
print("Before read:", vars(rectangle14))
assert rectangle14.perimeter == 14
print("After first read:", vars(rectangle14))
assert rectangle14.perimeter == 14
assert rectangle14.compute_calls == 1
print("Compute calls:", rectangle14.compute_calls)

Before read: {'compute_calls': 0, '__dimension_width': 3, '__dimension_height': 4}
After first read: {'compute_calls': 1, '__dimension_width': 3, '__dimension_height': 4, '__cache_perimeter': 14}
Compute calls: 1


### Step 4 — Change a dependency

A successful dimension assignment invalidates the dependent result. The cache is rebuilt on the next read, not immediately during assignment.

In [40]:
rectangle14.width = 10
print("After width assignment:", vars(rectangle14))
assert "__cache_perimeter" not in vars(rectangle14)
assert rectangle14.perimeter == 28
assert rectangle14.compute_calls == 2
print("New perimeter:", rectangle14.perimeter)

After width assignment: {'compute_calls': 1, '__dimension_width': 10, '__dimension_height': 4}
New perimeter: 28


### Step 5 — A rejected update must not invalidate good state

Our setter validates before touching storage or cache. As a result, a failed update preserves both the dimension and its valid cached perimeter.

In [41]:
before14 = dict(vars(rectangle14))
with expecting(ValueError):
    rectangle14.height = -1
assert vars(rectangle14) == before14
assert rectangle14.perimeter == 28
assert rectangle14.compute_calls == 2
with expecting(AttributeError):
    rectangle14.perimeter = 999
print("Rejected updates left state unchanged.")

Expected ValueError: height must be positive
Expected AttributeError: perimeter is a derived, read-only value
Rejected updates left state unchanged.


### Step 6 — Check instance isolation and lifetime

Each rectangle owns its own cached value and dimensions. The descriptor objects hold only metadata and no strong references to rectangles.

In [42]:
other14 = Rectangle14(5, 6)
assert other14.perimeter == 22
assert other14.compute_calls == 1
assert rectangle14.perimeter == 28
other14_ref = weakref.ref(other14)
del other14
gc.collect()
print("Unreferenced rectangle collected?", other14_ref() is None)
assert other14_ref() is None

Unreferenced rectangle collected? True


**Integrated solution:** valid values are instance-specific; validation precedes mutation; the derived value is cached per instance; a successful dependency change invalidates it; a failed change preserves it; and class-held descriptors do not retain managed objects. This is an appropriate small-model design, not a universal cache framework: multiple dependent values, concurrent mutation, slotted objects, and complex transactions require additional rules.

---

# Consolidated verification

The main conclusions are executable assertions below. Re-run the entire notebook from a clean kernel if you changed intermediate definitions. Each test targets a different property of descriptors rather than merely checking that a cell printed something.

In [43]:
assert Early.field.events == [("Early", "field")]
assert late_descriptor.events == [("Late", "field")]
assert item.number == 17
assert not hasattr(msg, "note")
assert safe5._code == "internal status" and safe5.code == "client code"
assert first6.session != second6.session
assert EquatableUser.token.tracked() == 0
assert item8_watch() is None and item8b_watch() is None
assert package9.weight == 12 and small9.height == 3
assert shallow10.tags is original10.tags
assert clone10.tags == source10.tags
assert invoice11.amount == 100
assert safe_blog12.tags == ("python", "descriptors")
assert wallet13.balance == 100
assert rectangle14.perimeter == 28 and rectangle14.compute_calls == 2
print("PASS: all 14 guided problems' final invariants")

PASS: all 14 guided problems' final invariants


# Further experiments (intentionally unsolved)

These are optional research extensions. They are **not** additional promised worked solutions.

1. Extend the identity store in Problem 7 to support a non-weak-referenceable `__slots__` owner. Decide whether doing so without modifying the owner is possible without a lifetime trade-off.
2. Write tests that explicitly demonstrate the different meanings of *equality* and *identity* in object storage.
3. Redesign Problem 8 around a callback that closes over its owner, and inspect the reference graph that causes retention.
4. Add `__deepcopy__` to Problem 10 and state a policy for mutable external values.
5. Add a `version` counter to Problem 14 so callers can tell when a derived value has become stale without reading it.
6. Add owner-level atomic updates to width and height together; ensure no observer sees a half-updated rectangle.
7. Write a test for the class access semantics of a descriptor typed as a dataclass field in your target Python version.

**Design checklist:** Where is the per-instance value stored? Does it preserve identity? What is held strongly? What happens when the value is missing? Can another operation bypass validation? What happens on copying, mutation, invalid input, class creation, and garbage collection?